# 0.Library

In [42]:
# general
import pandas as pd
import importlib
import joblib


# Paths
import sys
sys.path.append('../') 
from features import data_utils as du
import paths_data as paths

# Reload
importlib.reload(du)
importlib.reload(paths)

<module 'paths_data' from '/home/smira/myproject/detection_AD_with_VR_data/src/notebooks/../paths_data.py'>

# 1. Paths and Constants

In [43]:
# Read models path
models_paths_folder = paths.MODELS_DIR_SOL_ONE
best_model = "SelectKBest_SVR_20"

df_name_input = "cleaned_features_13_patients.csv"
df_path = paths.STAGE_THREE_DATA_PREPROCESSING / df_name_input

# outputs name and paths
df_name_output = f"predicted_13_patients_by_{best_model}.csv"
df_path_output = paths.PREDICTS_SOLTUION_ONE

# 2. Load best model and DF input

In [44]:
# Load the best model
loaded_model = joblib.load(models_paths_folder / f"{best_model}.pkl")
df = pd.read_csv(df_path)

X = df.copy()
X = X.drop(columns=['MoCA_Score']) 
X.head()

,Age,Help_Rating_out_of_5,Tutorial_total_reading_time,Tutorial_intensity_reading_time,Tutorial_total_count_hover,Tutorial_total_duration_hover,Tutorial_mean_duration_hover,Tutorial_max_duration_hover,Tutorial_total_count_press,Tutorial_median_duration_press,...,Memory_Yaw_std,Memory_Pitch_std,Memory_Roll_std,Memory_Yaw_range,Memory_Roll_range,Memory_dominant_hand_mean_speed,Memory_not_dominant_hand_mean_speed,Memory_dominant_hand_z_range,Memory_not_dominant_hand_y_range,Gender_Male
0,73.0,4,45.03,0.92,32,20.64,1.03,2.29,8,0.15,...,26.23,161.72,177.36,99.05,360.00,0.16,0.03,0.82,0.80,0
1,82.0,1,338.75,0.99,688,145.79,2.11,20.01,9,0.25,...,23.27,161.90,162.61,109.38,359.99,0.06,0.01,0.98,0.75,1
2,62.0,1,152.90,0.97,61,54.33,1.81,11.62,8,0.28,...,30.07,170.64,169.17,248.93,359.98,0.12,0.02,1.05,0.58,1
3,78.0,2,181.69,0.98,26,91.62,2.78,23.45,9,0.36,...,31.96,136.51,170.83,174.64,359.98,0.12,0.02,0.82,0.42,0
4,71.0,2,190.51,0.98,109,102.68,2.57,21.86,8,0.31,...,27.48,169.32,125.66,146.78,359.99,0.11,0.03,0.82,0.64,0


# 2. Predict MoCA score by best model

In [45]:
y_pred = loaded_model.predict(X)

In [46]:
y_pred

array([27.89979686, 26.26152295, 27.24930112, 27.09973687, 26.69299299,
       28.49428109, 26.28595373, 27.89979683, 28.0461207 , 25.93394207,
       27.87094211, 26.9006694 , 28.34560273])

In [47]:
df['Predicted_MoCA'] = y_pred
df['Predicted_MoCA'].head()

0    27.899797
1    26.261523
2    27.249301
3    27.099737
4    26.692993
Name: Predicted_MoCA, dtype: float64

# 3. Convert MoCa score to severity level

In [48]:
df['severity_level'] = df['MoCA_Score'].apply(du.severity_level_MoCA) # un comment this line when the df has MoCA value
df['predicted_severity_level'] = df['Predicted_MoCA'].apply(du.severity_level_MoCA)

In [49]:
#df[['MoCA_Score','severity_level','predicted_severity_level','Predicted_MoCA']] # uncomment this line when the Df has a MoCA score
df[['predicted_severity_level','Predicted_MoCA']]

,predicted_severity_level,Predicted_MoCA
0,Normal,27.899797
1,Normal,26.261523
2,Normal,27.249301
3,Normal,27.099737
4,Normal,26.692993
5,Normal,28.494281
6,Normal,26.285954
7,Normal,27.899797
8,Normal,28.046121
9,Mild impairment,25.933942


# 4. Save DF

In [50]:
df.to_csv(df_path_output / df_name_output, index=False)